# 1. test 데이터 성능 비교

In [1]:
from pathlib import Path

import pandas as pd
from ultralytics import YOLO


ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
DATA_YAML = ROOT / "data" / "yolo_subset" / "data.yaml"

MODELS = {
    "baseline": 640,
    "improved_v1": 960,
    "improved_v2": 960,
}


rows = []

for name, img_size in MODELS.items():
    model = YOLO(str(ROOT / "models" / name / "weights" / "best.pt"))

    result = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=img_size,
        conf=0.25,
        verbose=False,
        plots=False,
    )

    p, r = result.box.mp, result.box.mr

    rows.append([
        name, img_size, p, r, 2 * p * r / (p + r),
        result.box.map50, result.box.map75, result.box.map,
        result.speed["inference"],
    ])


columns = [
    "Model", "Image Size", "Precision", "Recall", "F1",
    "mAP50", "mAP75", "mAP50-95", "Inference(ms)"
]

df = pd.DataFrame(rows, columns=columns)

display(df.round(4))

Ultralytics 8.4.138  Python-3.11.16 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 185.959.1 MB/s, size: 122.1 KB)
val: Scanning C:\Users\PMS\Desktop\MS\project\small-drone-detection\data\yolo_subset\labels\test.cache... 733 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 733/733  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 46/46 8.6it/s 5.4s<0.2s
                   all        733        749      0.948      0.884      0.914      0.614
Speed: 2.9ms preprocess, 2.5ms inference, 0.0ms loss, 0.2ms postprocess per image
Ultralytics 8.4.138  Python-3.11.16 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
YOLO26s summary (fused): 122 layers, 9,465,567 parameters, 0 gradients, 20.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 609.2282.5 MB/s, size: 85.2 K

,Model,Image Size,Precision,Recall,F1,mAP50,mAP75,mAP50-95,Inference(ms)
0,baseline,640,0.9483,0.8838,0.9149,0.9143,0.7051,0.6139,2.4686
1,improved_v1,960,0.9661,0.9252,0.9452,0.9405,0.7500,0.6561,3.2162
2,improved_v2,960,0.9704,0.9372,0.9535,0.9419,0.7801,0.6852,2.9315


## 2. Test 성능 비교

| Model | Image Size | Precision | Recall | F1 | mAP50 | mAP75 | mAP50-95 | Inference(ms) |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Baseline | 640 | 0.9483 | 0.8838 | 0.9149 | 0.9143 | 0.7051 | 0.6139 | 2.47 |
| Improved V1 | 960 | 0.9661 | 0.9252 | 0.9452 | 0.9405 | 0.7500 | 0.6561 | 3.22 |
| Improved V2 | 960 | **0.9704** | **0.9372** | **0.9535** | **0.9419** | **0.7801** | **0.6852** | 2.93 |


## 3. 최종 판단

Baseline 실패 분석에서 가장 작은 객체군의 낮은 Recall과 bbox 위치 오차를 핵심 문제로 확인

- Baseline Validation: Q1 Recall **0.7489**, FN **123개**, FP **99개**
- V1: Input Size **640 → 960** 변경
    - Validation Q1 Recall **0.7489 → 0.8265**
    - FN **123 → 80개**, FP **99 → 75개**
- V2: Input Size 960 유지, Epoch **30 → 100** 증가
    - Validation 위치 부정확 FN **23 → 10개**
    - FP **75 → 51개**
    - mAP75 **0.7225 → 0.7703**
    - mAP50-95 **0.6202 → 0.6500**

최종 Test에서도 V2가 **Precision 0.9704 / Recall 0.9372 / F1 0.9535 / mAP50 0.9419 / mAP75 0.7801 / mAP50-95 0.6852**로 모든 주요 성능 지표에서 가장 높은 결과 확인

Baseline 대비 Test 성능은 Precision **+0.0221**, Recall **+0.0534**, F1 **+0.0386**, mAP75 **+0.0750**, mAP50-95 **+0.0713** 개선

Validation에서 확인한 **극소형 UAV 탐지와 bbox 정밀도 개선이 Test에서도 유지됨**을 확인

Input Size 증가로 추론 시간은 Baseline **2.47 ms → V2 2.93 ms**로 증가했지만 성능 개선 폭을 고려하여 **Improved V2를 최종 모델로 선정**


## 4. 잔여 오류 및 향후 개선 방향

### 잔여 오류

V2 Validation 실패 분석에서 전체 오류는 감소했지만 다음 문제가 남아 있음

| 잔여 문제 | 확인 결과 | 향후 개선 방향 |
|---|---|---|
| 극소형 UAV 미탐 | Q1 Recall 0.8311로 다른 크기 구간보다 낮음 | 극소형 객체 학습 샘플 보강 및 작은 객체 중심 데이터 증강 검토 |
| 완전 미검출 | FN 76개 중 미검출 41개 | 수목·복잡한 배경·낮은 대비 환경의 어려운 사례 추가 학습 |
| 낮은 신뢰도 | 낮은 신뢰도 FN 25개 | Validation 기준 Precision-Recall 확인 후 confidence threshold 최적화 |
| 배경 오탐 | 조류·비행기·건물 구조물·바위·그림자 등을 UAV로 검출 | 오탐 배경을 Hard Negative 데이터로 추가하여 배경 구분 능력 강화 |
| bbox 위치 오차 | 위치 부정확 FN 10개 잔존 | 극소형 객체의 annotation 일관성 확인 및 작은 객체 bbox 정밀도 추가 개선 |
| 데이터 일반화 | 동일·유사 촬영 환경의 데이터가 포함될 가능성 | 별도 촬영 환경 또는 외부 데이터셋을 이용한 추가 일반화 검증 |

### 개선 우선순위

1. **Hard Negative 보강**  
   실제 FP로 확인된 조류·비행기·건물 구조물·바위·그림자 이미지를 배경 학습 데이터로 추가

2. **극소형 UAV 데이터 보강**  
   Q1 크기 객체와 복잡한 배경의 미검출 사례를 중심으로 학습 데이터 보강

3. **Confidence Threshold 재검토**  
   낮은 신뢰도 FN과 FP의 균형을 Validation PR 결과로 비교하여 운영 threshold 선정

4. **독립 데이터 일반화 검증**  
   학습·Validation과 다른 촬영 환경에서 최종 V2 성능 추가 확인

추가 Epoch만 증가하는 방식보다 **잔여 실패 사례를 직접 데이터에 반영하는 방향의 개선 우선순위가 높음**
